# 🗺️ Station Query → Map

Ask a natural-language question about seismic stations (e.g. *"Stations within 5 km of Anchorage, Alaska"*), send it to an LLM via [`fdsn-agent`](../fdsn_agent) (built on top of `query.py`), and plot the stations it finds on a `folium` map.

**How it works:** `fdsn-agent` translates your question into an FDSN station-service query, runs it against IRIS, and hands back structured JSON (`result.data["stations"]`). This notebook turns that JSON into a `pandas` DataFrame and drops a marker on the map for each station.

**Requirements:**
- `fdsn_agent` installed (editable install from the sibling `fdsn_agent/` package: `pip install -e ../fdsn_agent`)
- An LLM backend reachable from `LLMConfig` — this notebook defaults to a local Ollama server (`ollama serve`, with the model pulled, e.g. `ollama pull gemma4`). Swap `LLMConfig.from_preset(...)` for `"anthropic"`, `"openai"`, etc. to use a different provider.

**How to play:** Run all cells, then type a question into the query box and click **Run Query** (or press Enter).

In [1]:
# --- Environment check: run this first ---
def _check():
    missing = []
    for pkg in ["fdsn_agent", "folium", "pandas", "ipywidgets"]:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)

    if missing:
        print("⚠️  Missing packages:", ", ".join(missing))
        if "fdsn_agent" in missing:
            print("    Run: pip install -e ../fdsn_agent   (editable install of the local package)")
        other = [p for p in missing if p != "fdsn_agent"]
        if other:
            print(f"    Run: pip install {' '.join(other)}")
    else:
        import fdsn_agent
        print("✅ fdsn_agent, folium, pandas, ipywidgets are all installed.")
        print(f"fdsn_agent: {fdsn_agent.__version__}")

_check()

✅ fdsn_agent, folium, pandas, ipywidgets are all installed.
fdsn_agent: 0.1.0


In [2]:
from fdsn_agent import Agent, LLMConfig

# Local Ollama by default -- swap the preset/model to use a different provider
# (see LLMConfig.from_preset in fdsn_agent_DOCS.md for "anthropic", "openai", etc.)
cfg = LLMConfig.from_preset("ollama", model="gemma4")
agent = Agent(cfg)

In [3]:
import folium
import pandas as pd


def get_station_dataframe(result):
    """Turn an fdsn_station AgentResult into a deduplicated station DataFrame.

    Raises ValueError if the LLM routed the query to a different tool (e.g. it
    read the question as an earthquake or waveform request instead of a
    station search), or if the station search came back empty.

    Deduplicates on (network, station, latitude, longitude): the FDSN station
    service returns one row per deployment epoch, so a station with several
    instrument swaps over its lifetime otherwise produces multiple overlapping
    markers at the same coordinates.
    """
    if result.tool_called != "fdsn_station":
        raise ValueError(
            f"Expected the fdsn_station tool, but the LLM called {result.tool_called!r} "
            "instead. Try rephrasing the query to make it clearer you're asking about "
            "station metadata rather than earthquakes or waveforms."
        )

    stations = result.data.get("stations", [])
    if not stations:
        raise ValueError("No stations returned for this query -- try widening it.")

    df = pd.DataFrame(stations)
    df = df.drop_duplicates(subset=["Network", "Station", "Latitude", "Longitude"])
    df = df.rename(columns=str.lower).reset_index(drop=True)
    for col in ("latitude", "longitude", "elevation"):
        df[col] = df[col].astype(float)
    return df


def build_station_map(df):
    """Build a folium map with a marker per station, auto-fit to their extent."""
    center = [df["latitude"].mean(), df["longitude"].mean()]
    m = folium.Map(location=center, tiles="OpenStreetMap")

    for _, row in df.iterrows():
        popup_html = (
            f"<b>{row['network']}.{row['station']}</b><br>"
            f"{row.get('sitename', '')}<br>"
            f"Elevation: {row['elevation']:.0f} m<br>"
            f"{row.get('starttime', '?')} – {row.get('endtime', '?')}"
        )
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=5,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"{row['network']}.{row['station']}",
            color="#1f77b4",
            fill=True,
            fill_opacity=0.7,
        ).add_to(m)

    bounds = [
        [df["latitude"].min(), df["longitude"].min()],
        [df["latitude"].max(), df["longitude"].max()],
    ]
    m.fit_bounds(bounds)
    return m


print("Functions loaded.")

Functions loaded.


In [4]:
import ipywidgets as widgets
from IPython.display import display


def on_run_clicked(_btn):
    query_text = query_input.value.strip()
    output.clear_output()

    if not query_text:
        with output:
            print("⚠️ Please enter a query.")
        return

    run_btn.disabled = True
    with output:
        print(f"🔎 Querying: {query_text}")

    try:
        result = agent.query(query_text)
    except Exception as exc:
        run_btn.disabled = False
        with output:
            print(f"❌ Query failed: {exc}")
        return

    with output:
        print(f"Tool called: {result.tool_called}({result.tool_params})")
        print(f"Summary: {result.summary}\n")

    try:
        station_df = get_station_dataframe(result)
    except ValueError as exc:
        run_btn.disabled = False
        with output:
            print(f"❌ {exc}")
        return

    run_btn.disabled = False
    with output:
        print(f"{len(station_df)} unique station(s) found.")
        display(station_df.head())
        display(build_station_map(station_df))


query_input = widgets.Text(
    value="Stations within 5 km of Anchorage, Alaska",
    description="Query:",
    placeholder="Ask about seismic stations...",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "initial"},
)
run_btn = widgets.Button(description="Run Query", button_style="success", icon="search")
run_btn.on_click(on_run_clicked)
query_input.on_submit(on_run_clicked)  # Enter key also runs the query

output = widgets.Output()

display(widgets.HBox([query_input, run_btn]), output)

/var/folders/v7/q9ywxgv10gl_hchl406lpn0c0000gp/T/ipykernel_66353/228121425.py:54: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  query_input.on_submit(on_run_clicked)  # Enter key also runs the query


Output()